# 03 — External datasets ingest and preparation

This notebook ingests external datasets and prepares them for integration with Google mobility analyses:
- BTS Daily Mobility Statistics — National and State (CSV)
- Hurricanes in USA 2019–2024 (CSV/Excel)
- State Population Estimates 2019–2024 (US Census Bureau CSVs)

Outputs: cleaned/typed Parquet files under `data/processed/`.

In [1]:
from hurricane_mobility.config import (
    BTS_CSV, HURRICANES_CSV, POP_2019_CSV, POP_2024_CSV,
    PROCESSED_DIR, ensure_dirs,
)
from hurricane_mobility.loaders import load_bts, load_hurricanes, load_population
from hurricane_mobility.features import compute_distance_band_shares
from hurricane_mobility.lookups import STATE_NAME_TO_CODE, STATE_CODE_TO_NAME

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

ensure_dirs()

In [2]:
# === BTS Daily Mobility ===
bts = load_bts(BTS_CSV)
print(f'BTS rows (state level): {len(bts):,}')

bts_tidy = compute_distance_band_shares(bts)

out_bts = PROCESSED_DIR / 'bts_state_daily.parquet'
bts_tidy.to_parquet(out_bts, index=False)
print(f'Wrote {out_bts}')

bts_tidy[[
    'date', 'State Postal Code', 'Number of Trips',
    'short_distance_share', 'medium_distance_share', 'long_distance_share',
]].head(5)

BTS rows (state level): 98,073
Wrote /Users/yzc/Desktop/CHIP/hurricane/data/processed/bts_state_daily.parquet


,date,State Postal Code,Number of Trips,short_distance_share,medium_distance_share,long_distance_share
0,2024-04-06,AL,16896499,0.904106,0.082984,0.012910
1,2024-04-06,AK,3429687,0.936355,0.055961,0.007685
2,2024-04-06,AZ,25016622,0.924162,0.062540,0.013298
3,2024-04-06,AR,10534439,0.902091,0.084269,0.013640
4,2024-04-06,CA,146325299,0.923622,0.067375,0.009004


In [3]:
# === Hurricanes ===
hur = load_hurricanes(HURRICANES_CSV)
print(f'Hurricane rows: {len(hur)}')

out_hur = PROCESSED_DIR / 'hurricanes_2019_2024.parquet'
hur.to_parquet(out_hur, index=False)
print(f'Wrote {out_hur}')

hur.head(5)

Hurricane rows: 19
Wrote /Users/yzc/Desktop/CHIP/hurricane/data/processed/hurricanes_2019_2024.parquet


,year,storm_name,landfall_state,state_code,landfall_date,start_date,end_date
0,2019,Barry,Louisiana,LA,2019-07-13,2019-07-06,2019-07-20
1,2019,Dorian,North Carolina,NC,2019-09-06,2019-08-30,2019-09-13
2,2020,Hanna,Texas,TX,2020-07-25,2020-07-18,2020-08-01
3,2020,Isaias,North Carolina,NC,2020-08-03,2020-07-27,2020-08-10
4,2020,Laura,Louisiana,LA,2020-08-27,2020-08-20,2020-09-03


In [4]:
# === Population (US Census Bureau CSVs: 2019 + 2020-2024) ===
pop = load_population(POP_2019_CSV, POP_2024_CSV)
print(f'Population rows: {len(pop)}')
assert len(pop) == 306, f'Expected 306 rows (51 states x 6 years), got {len(pop)}'

out_pop = PROCESSED_DIR / 'state_population_2019_2024.parquet'
pop.to_parquet(out_pop, index=False)
print(f'Wrote {out_pop}')

pop.head(10)

Population rows: 306
Wrote /Users/yzc/Desktop/CHIP/hurricane/data/processed/state_population_2019_2024.parquet


,state_code,state,year,population
0,AL,Alabama,2019,4903185
1,AK,Alaska,2019,731545
2,AZ,Arizona,2019,7278717
3,AR,Arkansas,2019,3017804
4,CA,California,2019,39512223
5,CO,Colorado,2019,5758736
6,CT,Connecticut,2019,3565287
7,DE,Delaware,2019,973764
8,DC,District Of Columbia,2019,705749
9,FL,Florida,2019,21477737


In [5]:
# Verify lookup utilities
print(f'States in lookup: {len(STATE_NAME_TO_CODE)}')
print('Sample:', list(STATE_NAME_TO_CODE.items())[:5])
print('Reverse:', list(STATE_CODE_TO_NAME.items())[:5])

States in lookup: 51
Sample: [('Alabama', 'AL'), ('Alaska', 'AK'), ('Arizona', 'AZ'), ('Arkansas', 'AR'), ('California', 'CA')]
Reverse: [('AL', 'Alabama'), ('AK', 'Alaska'), ('AZ', 'Arizona'), ('AR', 'Arkansas'), ('CA', 'California')]
